# The Headline Shift: Measuring Ideology and Sentiment in U.S. Online News Headlines Over Time

This notebook walks through the full analysis pipeline:
1. Data loading and exploration
2. Ideology classification (baseline + transformer)
3. Sentiment analysis
4. Time-series trends and visualizations

**Note:** Run `python data/download_data.py` and `python run_pipeline.py` first to generate the data and models.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline

## 1. Data Exploration

In [ ]:
from src.data_loader import load_qbias, split_qbias, load_headlines

# QBias labeled data
qbias = load_qbias()
print(f'QBias: {len(qbias)} headlines')
display(qbias['label'].value_counts())
display(qbias.head())

In [ ]:
# Headline corpus
headlines = load_headlines()
print(f'Headlines: {len(headlines)}')
display(headlines['publication'].value_counts())
display(headlines.head())

In [ ]:
# Headlines per year per publication
fig, ax = plt.subplots(figsize=(12, 5))
ct = headlines.groupby(['year', 'publication']).size().unstack(fill_value=0)
ct.plot.bar(stacked=True, ax=ax, colormap='Set2')
ax.set_title('Headlines per Year by Publication')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 2. Ideology Classification

In [ ]:
# Load the scored results (generated by run_pipeline.py)
import os
from src.utils import RESULTS_DIR

scored_path = os.path.join(RESULTS_DIR, 'scored_headlines.csv')
if os.path.exists(scored_path):
    scored = pd.read_csv(scored_path, parse_dates=['date'])
    print(f'Loaded {len(scored)} scored headlines')
    display(scored.head())
else:
    print('Run `python run_pipeline.py` first to generate scored data.')
    scored = None

In [ ]:
if scored is not None:
    # Ideology distribution per publication
    fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
    for idx, pub in enumerate(['CNN', 'Fox News', 'Washington Post', 'New York Times']):
        sub = scored[scored['publication'] == pub]
        sub['ideology_score'].value_counts().reindex(['left', 'center', 'right']).plot.bar(
            ax=axes[idx], color=['#2166AC', '#F7F7F7', '#B2182B'],
            edgecolor='gray'
        )
        axes[idx].set_title(pub)
        axes[idx].set_xlabel('')
    fig.suptitle('Ideology Label Distribution', fontsize=14)
    plt.tight_layout()
    plt.show()

## 3. Sentiment Analysis

In [ ]:
if scored is not None:
    # Sentiment stats
    print('Sentiment Summary by Publication:')
    display(scored.groupby('publication')[['sentiment_score', 'emotionality']].describe().round(3))

In [ ]:
if scored is not None:
    fig, ax = plt.subplots(figsize=(10, 5))
    for pub in ['CNN', 'Fox News', 'Washington Post', 'New York Times']:
        sub = scored[scored['publication'] == pub]
        sub['sentiment_score'].hist(ax=ax, alpha=0.5, bins=50, label=pub)
    ax.legend()
    ax.set_title('Sentiment Score Distributions')
    ax.set_xlabel('VADER Compound Score')
    plt.tight_layout()
    plt.show()

## 4. Time-Series Analysis

In [ ]:
if scored is not None:
    from src.time_series import (
        plot_ideology_trends, plot_sentiment_trends,
        plot_emotionality_trends, plot_ideology_by_topic,
        plot_sentiment_distribution, plot_ideology_distribution,
        plot_election_effect
    )
    
    print('Generating all plots...')
    plot_ideology_trends(scored)
    plot_sentiment_trends(scored)
    plot_emotionality_trends(scored)
    plot_ideology_by_topic(scored)
    plot_sentiment_distribution(scored)
    plot_ideology_distribution(scored)
    plot_election_effect(scored)
    print('Done! Check outputs/plots/')

## 5. Key Findings

After running the pipeline, summarize findings here:

- **Ideology trends**: Do publications show measurable shifts over time?
- **Sentiment**: Are headlines becoming more negative/emotional?
- **Election effects**: Do election years correspond to more emotionally charged headlines?
- **Topic variation**: Does ideology scoring vary by topic area?

See the generated plots in `outputs/plots/` for visual evidence.